# Sales & Revenue Analysis Dashboard\nProject 2 — Thiranex Data Analytics Internship\n\nDataset: UCI Online Retail. This notebook documents loading, cleaning, KPI creation and exploratory analysis used by the dashboard.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\n\npath = '../data/raw/Online Retail.xlsx'\ndf = pd.read_excel(path)\ndf.head()

In [ ]:
df.info()\n\ndf.isna().sum()

## Cleaning\nCancelled invoices are identified by InvoiceNo beginning with `C`. For the sales dashboard, cancelled invoices, non-positive quantities, non-positive prices and invalid dates are excluded. Missing product descriptions are labeled `Unknown Product`.

In [ ]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.strip()\ndf['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')\ndf['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')\ndf['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')\ndf['Description'] = df['Description'].fillna('Unknown Product')\ndf['IsCancellation'] = df['InvoiceNo'].str.upper().str.startswith('C')\nsales = df[(~df['IsCancellation']) & (df['InvoiceDate'].notna()) & (df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()\nsales['Revenue'] = sales['Quantity'] * sales['UnitPrice']\nsales['YearMonth'] = sales['InvoiceDate'].dt.to_period('M').dt.to_timestamp()\nsales.shape

In [ ]:
kpis = {\n    'Total Revenue': sales['Revenue'].sum(),\n    'Total Quantity': sales['Quantity'].sum(),\n    'Total Orders': sales['InvoiceNo'].nunique(),\n    'Unique Customers': sales['CustomerID'].dropna().nunique(),\n}\nkpis['Average Order Value'] = kpis['Total Revenue'] / kpis['Total Orders']\nkpis

In [ ]:
monthly = sales.groupby('YearMonth', as_index=False)['Revenue'].sum()\nmonthly.plot(x='YearMonth', y='Revenue', figsize=(12,5), marker='o', legend=False)\nplt.title('Monthly Revenue Trend')\nplt.ylabel('Revenue (£)')\nplt.show()

In [ ]:
top_products = sales.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)\ntop_products.sort_values().plot(kind='barh', figsize=(10,6))\nplt.title('Top 10 Products by Revenue')\nplt.xlabel('Revenue (£)')\nplt.show()

In [ ]:
top_countries = sales.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(15)\ntop_countries.sort_values().plot(kind='barh', figsize=(10,6))\nplt.title('Top Countries by Revenue')\nplt.xlabel('Revenue (£)')\nplt.show()

## Dashboard handoff\nThe cleaned table can be loaded into Power BI, Excel or the included Streamlit app. Use the DAX measures in `docs/DAX_Measures.md` if building the dashboard in Power BI.